# Qwen2.5-14B Prompt-Injection Logit Payload Generator

This notebook generates float64 next-token logit payloads for a prompt-injection / prompt-formatting experiment.

Goal: fix `Qwen/Qwen2.5-14B-Instruct` and compare three user-defined prompt-formatting variants. The underlying model weights and loading mode stay fixed; only the prompt formatting changes.

You have full control over the three variants. Edit `PROMPT_FORMATTING_VARIANTS` near the sandbox/save cells before saving. One run of this notebook loads the 14B model once and creates 18 payloads total:

```text
3 prompt-formatting variants x 6 prompts = 18 payloads
```


## Step 1: Install Dependencies

This installs the Hugging Face loading stack. `bitsandbytes` is needed only if you choose `8bit` or `4bit` loading.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece safetensors

## Step 2: Check Your GPU

This does not change anything. It just shows which GPU Colab gave you.

In [ ]:
!nvidia-smi

## Step 3: Choose and Safely Load the Fixed Model

The model is fixed for this experiment. Load it once. Prompt-formatting variants are defined in the next step, so you can edit and rerun them without reloading the model.


In [ ]:
# Optional storage/memory cleanup before loading a model.
# This cell is self-contained so the notebook can run cleanly from a fresh runtime.
import gc

for variable_name in ("model", "tokenizer"):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as exc:
    print("Torch cleanup skipped:", repr(exc))

# Clear Hugging Face and Torch caches from the default Colab locations.
# Use this when you intentionally want to free storage before a different model run.
!rm -rf /root/.cache/huggingface
!rm -rf /root/.cache/torch
!rm -rf /content/.cache/huggingface
!rm -rf /content/.cache/torch

# Clear pip/cache/temp junk.
!rm -rf /root/.cache/pip
!rm -rf /tmp/*

# Optional: remove local model output/logit files only after downloading them.
# !rm -f /content/*.pt
# !rm -f /content/*.zip

# Show remaining disk use.
!du -h --max-depth=2 /root 2>/dev/null | sort -hr | head -20
!du -h --max-depth=2 /content 2>/dev/null | sort -hr | head -20
!df -h


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_KEY = "14b"      # Fixed for this experiment.
LOAD_MODE = "8bit"    # Keep fixed unless intentionally testing a separate confound.

MODELS = {
    "14b": "Qwen/Qwen2.5-14B-Instruct",
}

def cleanup_loaded_model():
    """Release the currently loaded model/tokenizer, if they exist, before reloading."""
    import gc
    global model, tokenizer

    if "model" in globals():
        del model
    if "tokenizer" in globals():
        del tokenizer

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_selected_model(model_key, load_mode):
    """
    Safely load the fixed Qwen2.5-14B model and return (model, tokenizer, model_id).

    For the prompt-injection experiment, MODEL_KEY should remain fixed. The
    compared conditions are actual prompt wrappers applied during chat formatting,
    not different model checkpoints.
    """
    cleanup_loaded_model()

    if model_key not in MODELS:
        raise ValueError(f"MODEL_KEY must be one of {list(MODELS)}")
    selected_model_id = MODELS[model_key]
    quantization_config = None
    torch_dtype = None

    if load_mode == "bf16":
        torch_dtype = torch.bfloat16
    elif load_mode == "fp16":
        torch_dtype = torch.float16
    elif load_mode == "8bit":
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    elif load_mode == "4bit":
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    else:
        raise ValueError("LOAD_MODE must be 'bf16', 'fp16', '8bit', or '4bit'.")

    selected_tokenizer = AutoTokenizer.from_pretrained(selected_model_id)
    selected_model = AutoModelForCausalLM.from_pretrained(
        selected_model_id,
        device_map="auto",
        torch_dtype=torch_dtype,
        quantization_config=quantization_config,
    )

    selected_model.eval()
    print("Fixed model:", selected_model_id)
    print("Load mode:", load_mode)
    return selected_model, selected_tokenizer, selected_model_id


model, tokenizer, model_id = load_selected_model(MODEL_KEY, LOAD_MODE)


## Step 4: Define Float64 Logit Extraction and Save Helpers

The main helper is `save_next_token_logit_payload(...)`. Give it a context and filename, and it saves everything needed for later analysis: CPU float64 next-token logits, the plain context, the prompt-injection wrapper, the chat-formatted context, the formatted token ID sequence, and lightweight model metadata.

Precision convention:

- `payload["logits"]` is saved as a CPU float64 tensor.
- Float64 is a lossless widening conversion for the fp16, bf16, and fp32 logit values normally emitted by Hugging Face LLM inference.
- This does not recover precision that was never present inside quantized/bf16/fp16 inference; it simply prevents the notebook from introducing additional precision loss after extraction.


In [ ]:
from pathlib import Path


PAYLOAD_SCHEMA_VERSION = "next_token_logits_float64_v1"


def _logits_to_float64_cpu(raw_logits):
    """Return a CPU float64 copy of a one-dimensional logit vector."""
    if raw_logits.ndim != 1:
        raise ValueError("Expected a one-dimensional next-token logit vector")
    return raw_logits.detach().cpu().to(dtype=torch.float64).contiguous()


def messages_for_prompt_injection(context, prompt_injection_key=None):
    """Build chat messages for one context under one prompt-injection wrapper.

    A variant with `system_prompt=None` uses only a user message, allowing Qwen's
    normal chat-template behavior. Variants with a system prompt prepend that
    actual instruction before the user context.
    """
    if prompt_injection_key is None:
        prompt_injection_key = SANDBOX_PROMPT_FORMATTING_KEY
    if prompt_injection_key not in PROMPT_FORMATTING_VARIANTS:
        raise ValueError(f"Unknown prompt injection key: {prompt_injection_key}")

    injection_text = PROMPT_FORMATTING_VARIANTS[prompt_injection_key]["system_prompt"]
    if injection_text is None:
        return [{"role": "user", "content": context}]
    return [
        {"role": "system", "content": injection_text},
        {"role": "user", "content": context},
    ]


def format_context_for_qwen_chat(tokenizer, context, prompt_injection_key=None):
    """Convert a plain context into the exact chat-formatted string Qwen sees."""
    messages = messages_for_prompt_injection(context, prompt_injection_key=prompt_injection_key)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def get_next_token_logits(model, tokenizer, context, prompt_injection_key=None):
    """Run the model once and return the next-token logit vector as CPU float64."""
    formatted_context = format_context_for_qwen_chat(tokenizer, context, prompt_injection_key=prompt_injection_key)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    model.eval()
    with torch.inference_mode():
        outputs = model(**inputs)

    return _logits_to_float64_cpu(outputs.logits[0, -1, :])


def build_next_token_logit_payload(model, tokenizer, context, *, model_id=None, load_mode=None, prompt_injection_key=None):
    """Build the complete saved object for one context without writing it to disk."""
    if prompt_injection_key is None:
        prompt_injection_key = SANDBOX_PROMPT_FORMATTING_KEY

    formatted_context = format_context_for_qwen_chat(tokenizer, context, prompt_injection_key=prompt_injection_key)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    model.eval()
    with torch.inference_mode():
        outputs = model(**inputs)

    logits = _logits_to_float64_cpu(outputs.logits[0, -1, :])
    input_ids = inputs.input_ids[0].detach().cpu().long().contiguous()

    return {
        "payload_schema_version": PAYLOAD_SCHEMA_VERSION,
        "logits": logits,
        "context": context,
        "prompt_injection_key": prompt_injection_key,
        "prompt_injection_text": PROMPT_FORMATTING_VARIANTS[prompt_injection_key]["system_prompt"],
        "prompt_formatting_label": PROMPT_FORMATTING_VARIANTS[prompt_injection_key]["label"],
        "formatted_context": formatted_context,
        "formatted_context_token_ids": input_ids,
        "model_id": model_id,
        "model_key": MODEL_KEY,
        "load_mode": load_mode,
        "logits_dtype": str(logits.dtype),
        "logits_shape": tuple(logits.shape),
        "vocab_size": int(logits.shape[0]),
        "formatted_context_length_tokens": int(input_ids.shape[0]),
    }


def save_next_token_logit_payload(model, tokenizer, context, filename, *, model_id=None, load_mode=None, prompt_injection_key=None):
    """Save CPU float64 logits plus the prompt-injection wrapper metadata."""
    path = Path(filename)
    if path.suffix == "":
        path = path.with_suffix(".pt")

    path.parent.mkdir(parents=True, exist_ok=True)
    payload = build_next_token_logit_payload(
        model,
        tokenizer,
        context,
        model_id=model_id,
        load_mode=load_mode,
        prompt_injection_key=prompt_injection_key,
    )
    payload["saved_path"] = str(path)
    torch.save(payload, path)
    print(f"Saved float64 logits and context tokens to {path}")
    return payload


def save_many_next_token_logit_payloads(model, tokenizer, context_filename_pairs, *, model_id=None, load_mode=None, prompt_injection_key=None):
    """Save complete payloads for several contexts in one loop."""
    saved = {}
    for context, filename in context_filename_pairs:
        payload = save_next_token_logit_payload(
            model,
            tokenizer,
            context,
            filename,
            model_id=model_id,
            load_mode=load_mode,
            prompt_injection_key=prompt_injection_key,
        )
        saved[filename] = payload
    return saved


def top_k_logit_tokens(logits, tokenizer, k=10):
    """Return the k highest-logit tokens as IDs, strings, logits, and base probabilities."""
    if logits.ndim != 1:
        raise ValueError("logits must be a one-dimensional tensor")
    if k <= 0:
        raise ValueError("k must be positive")

    k = min(int(k), int(logits.shape[0]))
    logits_cpu = logits.detach().cpu().to(dtype=torch.float64)
    shifted_logits = logits_cpu - torch.max(logits_cpu)
    probs_cpu = torch.exp(shifted_logits)
    probs_cpu = probs_cpu / torch.sum(probs_cpu)
    values, token_ids = torch.topk(logits_cpu, k=k)

    rows = []
    for token_id, value in zip(token_ids.tolist(), values.tolist()):
        rows.append({
            "token_id": token_id,
            "token_string": tokenizer.decode([token_id]),
            "logit": float(value),
            "base_probability": float(probs_cpu[token_id]),
        })
    return rows


def print_top_k_logit_tokens(logits, tokenizer, k=10):
    """Print top_k_logit_tokens(...) in a compact notebook-friendly table."""
    rows = top_k_logit_tokens(logits, tokenizer, k=k)
    for rank, row in enumerate(rows, start=1):
        token_text = repr(row["token_string"])
        print(f"{rank:>2}. id={row['token_id']:<8} logit={row['logit']:>12.8g} prob={row['base_probability']:.12g} token={token_text}")
    return rows


## Step 5: Define Prompt-Formatting Variants

Edit this cell freely. It does not reload the model. Keep exactly three variants for the downstream `3 x 3` heatmap.


In [ ]:
# You control these. Keep exactly 3 variants for the downstream 3 x 3 heatmap.
# These are the final prompt-formatting variants used by the prompt-injection
# experiment. If you change any system prompt here, regenerate all 18 payloads
# and update context_token_tracker.md.
#
# `system_prompt=None` means no explicit extra system message; Qwen's chat
# template automatically supplies its default system prompt.
PROMPT_FORMATTING_VARIANTS = {
    "variant_1": {
        "label": "PI Variant 1",
        "system_prompt": None,
    },
    "variant_2": {
        "label": "PI Variant 2",
        "system_prompt": "You are a helpful assistant",
    },
    "variant_3": {
        "label": "PI Variant 3",
        "system_prompt": "Amadeus",
    },
}

# Used only by the sandbox cells. The save cell generates every variant above.
SANDBOX_PROMPT_FORMATTING_KEY = "variant_1"

if len(PROMPT_FORMATTING_VARIANTS) != 3:
    raise ValueError("Keep exactly 3 prompt-formatting variants for the downstream 3 x 3 heatmap.")
if SANDBOX_PROMPT_FORMATTING_KEY not in PROMPT_FORMATTING_VARIANTS:
    raise ValueError(f"SANDBOX_PROMPT_FORMATTING_KEY must be one of {list(PROMPT_FORMATTING_VARIANTS)}")

print("model_id:", model_id)
print("sandbox_prompt_formatting_key:", SANDBOX_PROMPT_FORMATTING_KEY)
print("sandbox_prompt_formatting_label:", PROMPT_FORMATTING_VARIANTS[SANDBOX_PROMPT_FORMATTING_KEY]["label"])
print("sandbox_system_prompt:", repr(PROMPT_FORMATTING_VARIANTS[SANDBOX_PROMPT_FORMATTING_KEY]["system_prompt"]))
print("all_prompt_formatting_keys:", tuple(PROMPT_FORMATTING_VARIANTS.keys()))


## Step 6: Sandbox: Inspect Formatted Input and Generate a Response

Use this section to test one prompt-formatting variant before saving logits. It prints the exact raw string after chat formatting and generates one sampled next token. Change `SANDBOX_PROMPT_FORMATTING_KEY` in Step 5 to inspect a different variant.


In [ ]:
SANDBOX_CONTEXT = "Output a random binary sequence and nothing else. I should not be seeing words in your response."

sandbox_formatted_context = format_context_for_qwen_chat(tokenizer, SANDBOX_CONTEXT)
print(sandbox_formatted_context)

In [ ]:
sandbox_inputs = tokenizer(sandbox_formatted_context, return_tensors="pt").to(model.device)

with torch.no_grad():
    sandbox_output_ids = model.generate(
        **sandbox_inputs,
        max_new_tokens=1,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

sandbox_new_tokens = sandbox_output_ids[0, sandbox_inputs.input_ids.shape[-1]:]
print(tokenizer.decode(sandbox_new_tokens, skip_special_tokens=True))
print("token id:", sandbox_new_tokens.item())

## Step 7: Generate and Save All 18 Prompt-Formatting Payloads

This cell saves every variant in `PROMPT_FORMATTING_VARIANTS` in one pass. Keep exactly 3 variants for the downstream heatmap:

```text
3 variants x 6 prompts = 18 payloads
```


In [ ]:
PROMPTS_TO_SAVE = [
    (
        1,
        "Output a random binary sequence and nothing else. I should not be seeing words in your response.",
    ),
    (
        2,
        "Generate a RANDOM number from {zero, one}. Your response should be a single numerical digit. Nothing else. Try to be as unpredictable as possible.",
    ),
    (
        3,
        "Generate a highly random sequence of zero's and one's. Your response should be a string of numerical digits and nothing else. Do NOT output non-numerical text.",
    ),
    (
        4,
        "Simulate a completely random coin flip. Output the number zero if it's heads. Output the number one if it's tails. Your output should be a single numerical digit. DON'T GIVE THE SAME RESPONSE EVERY TIME. TRY TO MAKE IT A FIFTY-FIFTY CHANCE.",
    ),
    (
        5,
        "Generate a completely random comma-separated list of zero's and ones. Each entry in the list must be a numerical digit. Do not output anything else except this list.",
    ),
    (
        6,
        "OUTPUT THE NUMBER ZERO OR THE NUMBER ONE. MAKE IT A FIFTY-FIFTY CHANCE. YOUR RESPONSE SHOULD BE A NUMERICAL DIGIT AND NOTHING ELSE.",
    ),
]

SAVED_PAYLOAD_SPECS = []
for prompt_formatting_key in PROMPT_FORMATTING_VARIANTS:
    file_prefix = f"qwen25_14b_prompt_injection_{prompt_formatting_key}_{LOAD_MODE}"
    for prompt_index, context in PROMPTS_TO_SAVE:
        SAVED_PAYLOAD_SPECS.append({
            "prompt_formatting_key": prompt_formatting_key,
            "prompt_index": prompt_index,
            "context": context,
            "filename": f"{file_prefix}_prompt_{prompt_index}_logits.pt",
        })

saved_payloads = {}
for spec in SAVED_PAYLOAD_SPECS:
    payload = save_next_token_logit_payload(
        model,
        tokenizer,
        spec["context"],
        spec["filename"],
        model_id=model_id,
        load_mode=LOAD_MODE,
        prompt_injection_key=spec["prompt_formatting_key"],
    )
    saved_payloads[spec["filename"]] = payload

print(f"Saved {len(saved_payloads)} payloads.")


## Step 8: Load and Inspect the Saved Logit Payloads

Use this after saving the 18 payloads. It loads each file, verifies the saved metadata, prints the base probabilities for token `0` and token `1`, and prints the top-k highest-logit tokens by token ID and decoded string.


In [ ]:
TOP_K = 10
ZERO_TOKEN_ID = 15
ONE_TOKEN_ID = 16


def stable_softmax_float64(logits):
    logits64 = logits.detach().cpu().to(dtype=torch.float64)
    shifted = logits64 - torch.max(logits64)
    probs = torch.exp(shifted)
    return probs / torch.sum(probs)


loaded_payloads = {}
probability_summary = []

for spec in SAVED_PAYLOAD_SPECS:
    saved_file = spec["filename"]
    print("=" * 80)
    print("prompt_formatting_key:", spec["prompt_formatting_key"])
    print("prompt:", spec["prompt_index"])
    print("file:", saved_file)

    payload = torch.load(saved_file, map_location="cpu")
    loaded_payloads[saved_file] = payload

    loaded_logits = payload["logits"]
    loaded_context_token_ids = payload["formatted_context_token_ids"]
    loaded_probs = stable_softmax_float64(loaded_logits)
    p_zero = float(loaded_probs[ZERO_TOKEN_ID])
    p_one = float(loaded_probs[ONE_TOKEN_ID])
    probability_summary.append({
        "prompt_formatting_key": spec["prompt_formatting_key"],
        "prompt": spec["prompt_index"],
        "file": saved_file,
        "p_zero": p_zero,
        "p_one": p_one,
        "p_zero_plus_p_one": p_zero + p_one,
    })

    print("model_id:", payload.get("model_id"))
    print("load_mode:", payload.get("load_mode"))
    print("prompt_injection_key:", payload.get("prompt_injection_key"))
    print("prompt_formatting_label:", payload.get("prompt_formatting_label"))
    print("prompt_injection_text:", repr(payload.get("prompt_injection_text")))
    print("payload_schema_version:", payload.get("payload_schema_version"))
    print("logit shape:", tuple(loaded_logits.shape))
    print("logit dtype:", loaded_logits.dtype)
    print("logit device:", loaded_logits.device)
    print("formatted context token count:", len(loaded_context_token_ids))
    print("first 20 formatted context token ids:", loaded_context_token_ids[:20].tolist())
    print("P(token '0', id 15):", f"{p_zero:.12g}")
    print("P(token '1', id 16):", f"{p_one:.12g}")
    print("P('0') + P('1'):", f"{p_zero + p_one:.12g}")
    print(f"top {TOP_K} tokens by raw logit:")
    print_top_k_logit_tokens(loaded_logits, tokenizer, k=TOP_K)
    print()

print("=" * 80)
print("Probability summary for target tokens")
for row in probability_summary:
    print(
        f"{row['prompt_formatting_key']} prompt {row['prompt']}: "
        f"P(0)={row['p_zero']:.12g}, "
        f"P(1)={row['p_one']:.12g}, "
        f"P(0)+P(1)={row['p_zero_plus_p_one']:.12g}, "
        f"file={row['file']}"
    )
